In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("dataset_part_2.csv")
df.head()

,FlightNumber,PayloadMass,Orbit,LaunchSite,Flights,GridFins,Reused,Legs,Block,Longitude,Latitude,Class,Year
0,6,8117.574038,LEO,CCSFS SLC 40,1,False,False,False,1,-80.577366,28.561857,0,2010
1,7,8117.574038,LEO,CCSFS SLC 40,1,False,False,False,1,-80.577366,28.561857,0,2010
2,8,525.000000,LEO,CCSFS SLC 40,1,False,False,False,1,-80.577366,28.561857,0,2012
3,9,400.000000,ISS,CCSFS SLC 40,1,False,False,False,1,-80.577366,28.561857,0,2012
4,10,677.000000,ISS,CCSFS SLC 40,1,False,False,False,1,-80.577366,28.561857,0,2013


In [2]:
features = df[[
    "FlightNumber",
    "PayloadMass",
    "Orbit",
    "LaunchSite",
    "Flights",
    "GridFins",
    "Reused",
    "Legs",
    "Block",
    "Year"
]]

X = pd.get_dummies(features, columns=["Orbit", "LaunchSite"], drop_first=False)
y = df["Class"]

X.head()

,FlightNumber,PayloadMass,Flights,GridFins,Reused,Legs,Block,Year,Orbit_ES-L1,Orbit_GEO,...,Orbit_LEO,Orbit_MEO,Orbit_PO,Orbit_SO,Orbit_SSO,Orbit_TLI,Orbit_VLEO,LaunchSite_CCSFS SLC 40,LaunchSite_KSC LC 39A,LaunchSite_VAFB SLC 4E
0,6,8117.574038,1,False,False,False,1,2010,False,False,...,True,False,False,False,False,False,False,True,False,False
1,7,8117.574038,1,False,False,False,1,2010,False,False,...,True,False,False,False,False,False,False,True,False,False
2,8,525.000000,1,False,False,False,1,2012,False,False,...,True,False,False,False,False,False,False,True,False,False
3,9,400.000000,1,False,False,False,1,2012,False,False,...,False,False,False,False,False,False,False,True,False,False
4,10,677.000000,1,False,False,False,1,2013,False,False,...,False,False,False,False,False,False,False,True,False,False


In [3]:
transform = StandardScaler()
X_scaled = transform.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=2
)

print("Test size:", len(y_test))

Test size: 36


In [4]:
parameters = {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}
lr = LogisticRegression(max_iter=1000)

logreg_cv = GridSearchCV(lr, parameters, cv=10)
logreg_cv.fit(X_train, y_train)

print("Best Logistic Regression:", logreg_cv.best_params_)
print("Validation score:", logreg_cv.best_score_)

yhat_lr = logreg_cv.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, yhat_lr))
print(confusion_matrix(y_test, yhat_lr))

Best Logistic Regression: {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}
Validation score: 0.8947619047619048
Test accuracy: 0.8888888888888888
[[ 5  4]
 [ 0 27]]


In [5]:
parameters = {
    'kernel': ('linear', 'rbf', 'poly', 'rbf', 'sigmoid'),
    'C': np.logspace(-3, 3, 5),
    'gamma': np.logspace(-3, 3, 5)
}

svm = SVC()

svm_cv = GridSearchCV(svm, parameters, cv=10)
svm_cv.fit(X_train, y_train)

print("Best SVM:", svm_cv.best_params_)
print("Validation score:", svm_cv.best_score_)

yhat_svm = svm_cv.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, yhat_svm))
print(confusion_matrix(y_test, yhat_svm))

Best SVM: {'C': np.float64(0.03162277660168379), 'gamma': np.float64(0.001), 'kernel': 'linear'}
Validation score: 0.9157142857142857
Test accuracy: 0.8888888888888888
[[ 5  4]
 [ 0 27]]


In [6]:
parameters = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': [2, 4, 6, 8, 10],
    'max_features': ['auto', 'sqrt', 'log2'],
    'min_samples_leaf': [1, 2, 4],
    'min_samples_split': [2, 5, 10]
}

tree = DecisionTreeClassifier()

tree_cv = GridSearchCV(tree, parameters, cv=10)
tree_cv.fit(X_train, y_train)

print("Best Tree:", tree_cv.best_params_)
print("Validation score:", tree_cv.best_score_)

yhat_tree = tree_cv.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, yhat_tree))
print(confusion_matrix(y_test, yhat_tree))

Best Tree: {'criterion': 'gini', 'max_depth': 8, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'splitter': 'random'}
Validation score: 0.9228571428571428
Test accuracy: 0.8055555555555556
[[ 4  5]
 [ 2 25]]


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
1800 fits failed out of a total of 5400.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1800 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 436, in _validate_params
    validate_parameter_constraints(
  File "/usr/local/lib/python3.12/dist-packages/sklearn/u

In [7]:
parameters = {'n_neighbors': [1, 3, 5, 7, 9, 11]}
knn = KNeighborsClassifier()

knn_cv = GridSearchCV(knn, parameters, cv=10)
knn_cv.fit(X_train, y_train)

print("Best KNN:", knn_cv.best_params_)
print("Validation score:", knn_cv.best_score_)

yhat_knn = knn_cv.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, yhat_knn))
print(confusion_matrix(y_test, yhat_knn))

Best KNN: {'n_neighbors': 11}
Validation score: 0.9014285714285715
Test accuracy: 0.8888888888888888
[[ 5  4]
 [ 0 27]]


In [8]:
models = {
    "Logistic Regression": accuracy_score(y_test, yhat_lr),
    "SVM": accuracy_score(y_test, yhat_svm),
    "Decision Tree": accuracy_score(y_test, yhat_tree),
    "KNN": accuracy_score(y_test, yhat_knn)
}

best_model = max(models, key=models.get)

print("Model Accuracies:")
for model, acc in models.items():
    print(f"{model}: {acc:.4f}")

print("\nBest Model:", best_model)

Model Accuracies:
Logistic Regression: 0.8889
SVM: 0.8889
Decision Tree: 0.8056
KNN: 0.8889

Best Model: Logistic Regression
